# Step 41 Build Final No-Lag Fusion Observation Segments

## What This Notebook Does

This is the second public Stage-4 notebook. It preserves the original segment-construction workflow that:
- loads the per-run alignment outputs from Step 40
- combines the saved keep masks with a finite-row check
- concatenates the BOLD and EEG feature blocks into retained observation matrices
- writes one `.npy` file per retained segment
- writes the manifest and run-level summaries consumed by later HMM stages

## When To Run It

Run this notebook after `step40_align_eeg_to_bold_trs_and_build_keep_masks.ipynb`.

The public manuscript default is:
- `FEATURE_MODE = "nolags"`
- `MINLEN = 15`

An optional lagged `minlen15` branch can still be written for later provenance or model-comparison work, but it is not the main public dataset.

## Manuscript Linkage

- Main Methods 2.4
- Supplementary Methods 1.5
- Table S7 support
- final retained dataset manifest support

## Inputs Expected

- one completed Step-40 alignment output root, especially the `per_run/` products with keep masks, aligned TR features, and TR edges

## Outputs Written

- `hmm_segments_minlen15_nolags/segments_manifest.tsv`
- `hmm_segments_minlen15_nolags/qc/per_run_segments_minlen15.csv`
- per-segment `.npy` observation matrices for the canonical final dataset
- optional `hmm_segments_minlen15_lags/` outputs when requested

## Downstream Note

Later modeling stages use the canonical no-lag `minlen15` outputs from this notebook.


In [ ]:
# Step 0. Import Python modules and locate the active Stage-4 helper module

import sys
from pathlib import Path

STAGE4_DIR = Path.cwd()
candidate = Path.cwd() / "notebooks" / "4_alignment"
if not (STAGE4_DIR / "stage4_segment_helpers.py").exists() and candidate.exists():
    STAGE4_DIR = candidate

if not (STAGE4_DIR / "stage4_segment_helpers.py").exists():
    raise FileNotFoundError(
        f"Stage-4 helper not found: {STAGE4_DIR / 'stage4_segment_helpers.py'}"
    )

if str(STAGE4_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(STAGE4_DIR.resolve()))

from stage4_segment_helpers import build_segments_dataset


In [ ]:
# Step 1. User-editable roots and preserved segment settings
#
# ALIGNMENT_OUTPUT_DIR should point to the Step-40 output root.
# PER_RUN_ALIGNMENT_DIR is derived from it and holds the per-run aligned inputs.
# SEGMENTS_OUTPUT_DIR is the canonical manuscript dataset root for
# the no-lag minlen15 export used downstream.
# OPTIONAL_LAGGED_SEGMENTS_OUTPUT_DIR is only for the secondary lagged
# provenance branch if you choose to write it.

ALIGNMENT_OUTPUT_DIR = Path("<SET_ALIGNMENT_OUTPUT_DIR>")
PER_RUN_ALIGNMENT_DIR = ALIGNMENT_OUTPUT_DIR / "per_run"
SEGMENTS_OUTPUT_DIR = ALIGNMENT_OUTPUT_DIR / "hmm_segments_minlen15_nolags"
OPTIONAL_LAGGED_SEGMENTS_OUTPUT_DIR = ALIGNMENT_OUTPUT_DIR / "hmm_segments_minlen15_lags"

FEATURE_MODE = "nolags"
MINLEN = 15
TR_SEC = 2.1

WRITE_OPTIONAL_LAGGED_MINLEN15 = False
OPTIONAL_LAGGED_FEATURE_MODE = "lags"

MAKE_PLOTS = True


In [ ]:
# Step 2. Validate the configured inputs and print a short run summary

def assert_configured_path(path_value, label):
    path_str = str(path_value)
    if "<SET_" in path_str:
        raise ValueError(f"{label} is still using a placeholder path. Edit this notebook before running.")

assert_configured_path(ALIGNMENT_OUTPUT_DIR, "ALIGNMENT_OUTPUT_DIR")

print("Stage 4 / Step 41: build final fusion observation segments")
print(f"  Alignment output root:  {ALIGNMENT_OUTPUT_DIR}")
print(f"  Per-run input root:     {PER_RUN_ALIGNMENT_DIR}")
print(f"  Canonical output root:  {SEGMENTS_OUTPUT_DIR}")
print(f"  Canonical feature mode: {FEATURE_MODE}")
print(f"  Canonical min length:   {MINLEN} TR")
print(f"  Optional lagged build:  {WRITE_OPTIONAL_LAGGED_MINLEN15}")
print("  Downstream use:         later HMM stages read the canonical no-lag branch.")


In [ ]:
# Step 3. Build the canonical no-lag minlen15 dataset used by the public pipeline

canonical_result = build_segments_dataset(
    per_run_dir=PER_RUN_ALIGNMENT_DIR,
    feature_mode=FEATURE_MODE,
    minlen=MINLEN,
    tr_sec=TR_SEC,
    out_root=SEGMENTS_OUTPUT_DIR,
    make_plots=MAKE_PLOTS,
)

print("Saved canonical segment audit:", canonical_result["audit_csv"])
print("Saved canonical manifest:", canonical_result["manifest_tsv"])
print("Saved canonical run summary:", canonical_result["per_run_qc_csv"])

canonical_result["per_run_qc_df"].head(20)


In [ ]:
# Step 4. Optional: write a lagged minlen15 branch for provenance support only

optional_lagged_result = None
if WRITE_OPTIONAL_LAGGED_MINLEN15:
    optional_lagged_result = build_segments_dataset(
        per_run_dir=PER_RUN_ALIGNMENT_DIR,
        feature_mode=OPTIONAL_LAGGED_FEATURE_MODE,
        minlen=MINLEN,
        tr_sec=TR_SEC,
        out_root=OPTIONAL_LAGGED_SEGMENTS_OUTPUT_DIR,
        make_plots=MAKE_PLOTS,
    )
    print("Saved optional lagged manifest:", optional_lagged_result["manifest_tsv"])
else:
    print("WRITE_OPTIONAL_LAGGED_MINLEN15=False -> skipping lagged minlen15 export.")
